In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from matplotlib import transforms
import torch
from torch import nn
from torch.nn import functional as F
import torch.optim as optim
from tqdm.auto import tqdm
import copy
import scipy.linalg

# ---------- Params ----------
device_str = "cuda:0"
diffusion_steps = 1000
FID_SAMPLES = 1000000
N_base = 10**3
EPOCHS_BASE = 10000
LR_BASE = 1e-4

# ---------- Repro / Device ----------
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
device = torch.device(device_str if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ---------- Target Gaussian ----------
D = 2
sigma_ref = np.array([[2.0, 1.0], [1.0, 2.0]], dtype=np.float64)
mu_ref = np.zeros((D,), dtype=np.float64)

# ---------- Schedules ----------
def make_cosine_schedule(n_steps: int, s: float = 0.008, device=None):
    device = device or torch.device("cpu")
    t = torch.arange(0, n_steps, dtype=torch.float32, device=device)
    schedule = torch.cos((t / n_steps + s) / (1 + s) * torch.pi / 2) ** 2
    baralphas = schedule / schedule[0]
    betas = 1.0 - baralphas / torch.cat([baralphas[0:1], baralphas[:-1]])
    alphas = 1.0 - betas
    return {"alphas": alphas, "betas": betas, "baralphas": baralphas}

train_sched = make_cosine_schedule(diffusion_steps, device=device)

# ---------- Model & Utilities ----------
def noise(Xbatch: torch.Tensor, t_idx: torch.Tensor, baralphas: torch.Tensor):
    baralpha_t = baralphas[t_idx.squeeze(-1)].unsqueeze(-1)
    eps = torch.randn_like(Xbatch)
    noised = baralpha_t.sqrt() * Xbatch + (1.0 - baralpha_t).sqrt() * eps
    return noised, eps

class DiffusionBlock(nn.Module):
    def __init__(self, n: int):
        super().__init__()
        self.l = nn.Linear(n, n)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.relu(self.l(x))

class DiffusionModel(nn.Module):
    def __init__(self, nfeatures: int, nblocks: int = 4, nunits: int = 32):
        super().__init__()
        self.inb = nn.Linear(nfeatures + 1, nunits)
        self.mbs = nn.ModuleList([DiffusionBlock(nunits) for _ in range(nblocks)])
        self.out = nn.Linear(nunits, nfeatures)
    def forward(self, x: torch.Tensor, t: torch.Tensor, n_steps: int) -> torch.Tensor:
        if t.dtype not in (torch.float32, torch.float64):
            t = t.float()
        t_scaled = t / max(1, n_steps - 1)
        h = torch.hstack([x, t_scaled])
        h = self.inb(h)
        for block in self.mbs:
            h = block(h)
        return self.out(h)

@torch.no_grad()
def sample_diffusion(model: nn.Module, nsamples: int, nfeatures: int, n_steps: int, eta: float = 1.0, score_scale: float = 1.0) -> torch.Tensor:
    sched = make_cosine_schedule(n_steps, device=device)
    alphas, baralphas = sched["alphas"], sched["baralphas"]
    x = torch.randn((nsamples, nfeatures), device=device)

    for t in range(n_steps - 1, -1, -1):
        t_batch = torch.full((nsamples, 1), t, device=device, dtype=torch.long)
        eps_pred = model(x, t_batch, n_steps)
        
        bar_t = baralphas[t]
        bar_tm1 = baralphas[t-1] if t > 0 else torch.tensor(1.0, device=device)
        
        sqrt_bar_t = torch.sqrt(bar_t)
        sqrt_1m_bar_t = torch.sqrt(1.0 - bar_t)

        score = -eps_pred / (sqrt_1m_bar_t + 1e-9)
        score_scaled = score_scale * score
        
        x0_pred = (x + (1.0 - bar_t) * score_scaled) / (sqrt_bar_t + 1e-9)
        # x0_pred.clamp_(-3., 3.) # Clamping removed for more natural plots
        
        if t == 0:
            x = x0_pred
            break

        alpha_t = alphas[t]
        term1 = (1.0 - bar_tm1) / (1.0 - bar_t + 1e-9)
        term2 = 1.0 - alpha_t / (bar_t + 1e-9)
        sigma_t = eta * torch.sqrt(torch.clamp(term1 * term2, min=0.0))
        
        dir_coeff = torch.sqrt(torch.clamp(1.0 - bar_tm1 - sigma_t**2, min=0.0))
        
        mean = torch.sqrt(bar_tm1) * x0_pred + dir_coeff * eps_pred
        x = mean + (sigma_t * torch.randn_like(x) if (t > 0 and eta > 0.0) else 0.0)
    return x

def merge_models(base: nn.Module, other: nn.Module, w: float):
    m = copy.deepcopy(base)
    with torch.no_grad():
        for pm, pb, po in zip(m.parameters(), base.parameters(), other.parameters()):
            pm.copy_((1.0 + w) * pb - w * po)
    return m

# ---------- Metrics ----------
def calculate_fid(mu, sigma, mu_r, sigma_r) -> float:
    m = np.square(mu - mu_r).sum()
    s, _ = scipy.linalg.sqrtm(sigma @ sigma_r, disp=False)
    if not np.isfinite(s).all():
        return float('inf')
    return float(np.real(m + np.trace(sigma + sigma_r - 2.0 * s)))

def fid_of_model(model: nn.Module, nsamples: int, n_steps_infer: int) -> float:
    xs = sample_diffusion(model, nsamples, D, n_steps=n_steps_infer).cpu().numpy()
    if np.isnan(xs).any() or np.isinf(xs).any():
        return float('inf')
    return calculate_fid(xs.mean(0), np.cov(xs.T), mu_ref, sigma_ref)

# ===============================================
# ================== TRAIN BASE =================
# ===============================================
print("--- Training Base Model ---")
Xb = torch.from_numpy(np.random.multivariate_normal(mu_ref, sigma_ref, N_base).astype(np.float32))
base_model = DiffusionModel(D, nblocks=4).to(device)
opt = optim.Adam(base_model.parameters(), lr=LR_BASE)
loss_fn = nn.MSELoss()

for epoch in tqdm(range(EPOCHS_BASE), desc="Training Base"):
    for i in range(0, len(Xb), 2048):
        xb = Xb[i : i + 2048].to(device)
        t_idx = torch.randint(0, diffusion_steps, (len(xb), 1), device=device)
        noised, eps = noise(xb, t_idx, train_sched["baralphas"])
        pred = base_model(noised, t_idx, diffusion_steps)
        loss = loss_fn(pred, eps)
        opt.zero_grad()
        loss.backward()
        opt.step()
base_model.eval()
print("Base model training complete.")

# ===============================================
# === EXPERIMENT 1: FID vs. w ===
# ===============================================
print("\n--- Running Experiment 1: FID vs. w ---")
EPOCHS_LIST = [50, 250]
score_scales_fid = [0.9, 1.1]
LR_FT = 1e-5
N_aux = 10**3
ws_sweep = np.linspace(-2.0, 2.0, 101)
n_steps_eval = 20
results_fid = []

for s_scale in tqdm(score_scales_fid, desc="Sweeping s_scale for FID"):
    Xs = sample_diffusion(base_model, nsamples=N_aux, nfeatures=D, n_steps=n_steps_eval, score_scale=s_scale).cpu()
    aux_model = copy.deepcopy(base_model).train()
    opt_ft = optim.Adam(aux_model.parameters(), lr=LR_FT)
    max_epochs = max(EPOCHS_LIST)
    for epoch in range(1, max_epochs + 1):
        for i in range(0, len(Xs), 2048):
            xb = Xs[i : i + 2048].to(device)
            t_idx = torch.randint(0, diffusion_steps, (len(xb), 1), device=device)
            noised, eps = noise(xb, t_idx, train_sched["baralphas"])
            pred = aux_model(noised, t_idx, diffusion_steps)
            loss = loss_fn(pred, eps)
            opt_ft.zero_grad(); loss.backward(); opt_ft.step()
        if epoch in EPOCHS_LIST:
            aux_model.eval()
            for w in ws_sweep:
                merged_model = merge_models(base_model, aux_model, w)
                fid = fid_of_model(merged_model, nsamples=FID_SAMPLES, n_steps_infer=n_steps_eval)
                results_fid.append({'epoch': epoch, 's_scale': s_scale, 'w': w, 'fid': fid})
            aux_model.train()

df_fid = pd.DataFrame(results_fid)
df_fid.to_csv('fid_vs_w_results.csv', index=False)
print("Experiment 1 finished. Results saved to 'fid_vs_w_results.csv'")

# ===============================================
# === EXPERIMENT 2: Cosine Similarity vs. s_scale ===
# ===============================================
print("\n--- Running Experiment 2: Cosine Similarity vs. s_scale ---")

def _avg_grad_over_dataset(model, X, batch_size=2048, n_t_samples=16):
    model.train()
    loss_fn = nn.MSELoss()
    per_param_grads = [torch.zeros_like(p, device=device) for p in model.parameters()]
    n_batches = (len(X) + batch_size - 1) // batch_size
    for _ in range(n_t_samples):
        for i in range(0, len(X), batch_size):
            xb = X[i:i+batch_size].to(device)
            t_idx = torch.randint(0, diffusion_steps, (len(xb), 1), device=device)
            x_noised, eps = noise(xb, t_idx, train_sched["baralphas"])
            model.zero_grad()
            pred = model(x_noised, t_idx, diffusion_steps)
            loss = loss_fn(pred, eps)
            loss.backward()
            for acc, p in zip(per_param_grads, model.parameters()):
                if p.grad is not None: acc.add_(p.grad)
    for acc in per_param_grads: acc.div_(n_batches * n_t_samples)
    return [g.detach() for g in per_param_grads]

def _estimate_adam_preconditioner_diagonal(model, X, batch_size=2048, n_t_samples=16, betas=(0.9, 0.999), eps=1e-8):
    beta2 = betas[1]
    model.train()
    loss_fn = nn.MSELoss()
    v = [torch.zeros_like(p, device=device) for p in model.parameters()]
    steps = 0
    for _ in range(n_t_samples):
        for i in range(0, len(X), batch_size):
            xb = X[i:i+batch_size].to(device)
            t_idx = torch.randint(0, diffusion_steps, (len(xb), 1), device=device)
            x_noised, eps_t = noise(xb, t_idx, train_sched["baralphas"])
            model.zero_grad()
            pred = model(x_noised, t_idx, diffusion_steps)
            loss = loss_fn(pred, eps_t)
            loss.backward()
            for vv, p in zip(v, model.parameters()):
                if p.grad is not None: vv.mul_(beta2).addcmul_(p.grad, p.grad, value=(1.0 - beta2))
            steps += 1
    bc = 1.0 - (beta2 ** steps)
    for i in range(len(v)): v[i] = v[i] / bc
    return [1.0 / (torch.sqrt(vv) + eps) for vv in v]

# --- NEW HELPER FUNCTIONS FOR COSINE SIMILARITY ---
def _dot_with_P(g1_parts, g2_parts, P_diag_parts):
    s = 0.0
    for g1, g2, pd in zip(g1_parts, g2_parts, P_diag_parts):
        s += torch.sum(g1 * (pd * g2))
    return s.item()

def _norm_with_P(g_parts, P_diag_parts):
    norm_sq = 0.0
    for g, pd in zip(g_parts, P_diag_parts):
        norm_sq += torch.sum(g * (pd * g)) # This is <g, P g>
    return torch.sqrt(norm_sq).item()

S_SCALES_align = np.linspace(0.8, 1.2, 46)
N_pop = 100_000
N_syn = 100_000

X_pop = torch.from_numpy(np.random.multivariate_normal(mu_ref, sigma_ref, size=N_pop).astype(np.float32))
g_d_pop_parts = _avg_grad_over_dataset(base_model, X_pop)
P_adam_diag_parts = _estimate_adam_preconditioner_diagonal(base_model, X_pop)
norm_d = _norm_with_P(g_d_pop_parts, P_adam_diag_parts)

cosine_similarities = []
for s_scale in tqdm(S_SCALES_align, desc="Sweeping s_scale for Cosine Similarity"):
    X_syn = sample_diffusion(base_model, nsamples=N_syn, nfeatures=D, n_steps=n_steps_eval, score_scale=s_scale)
    g_s_parts = _avg_grad_over_dataset(base_model, X_syn)
    
    # Calculate cosine similarity
    inner_product = _dot_with_P(g_d_pop_parts, g_s_parts, P_adam_diag_parts)
    norm_s = _norm_with_P(g_s_parts, P_adam_diag_parts)
    cos_sim = inner_product / (norm_d * norm_s + 1e-9)
    cosine_similarities.append(cos_sim)

df_align = pd.DataFrame({'s_scale': S_SCALES_align, 'cosine_similarity': cosine_similarities})
df_align.to_csv('cosine_similarity_results.csv', index=False)
print("Experiment 2 finished. Results saved to 'cosine_similarity_results.csv'")

# ===============================================
# ================== PLOTTING ===================
# ===============================================
print("\n--- Plotting Results ---")
for epoch in sorted(df_fid['epoch'].unique()):
    plt.figure(figsize=(8, 6))
    epoch_df = df_fid[df_fid['epoch'] == epoch]
    for s_scale in sorted(epoch_df['s_scale'].unique()):
        subset = epoch_df[epoch_df['s_scale'] == s_scale]
        label = f"$\\zeta={s_scale:.2f}$ (mode-seeking)" if s_scale > 1.0 else f"$\\zeta={s_scale:.2f}$ (diversity-seeking)"
        plt.plot(subset['w'], np.log10(subset['fid']), label=label)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.title(f"FID vs. Merge Weight ($w$) after {epoch} FT Epochs")
    plt.xlabel("Merge Weight ($w$)")
    plt.ylabel("$\\log_{10}(\\mathrm{FID})$")
    plt.axvline(0, color='k', linestyle=':', linewidth=1, label='Base Model ($w=0$)')
    plt.legend(title="Source Sampler")
    plt.show()

plt.figure(figsize=(8, 6))
plt.plot(df_align['s_scale'], df_align['cosine_similarity'], marker='o', linestyle='-', markersize=4)
plt.axhline(0.0, color='k', linestyle="--", linewidth=1)
plt.xlabel("Sampler Score Scale ($\\zeta$)")
plt.ylabel("Cosine Similarity $\\cos(\\theta)$")
plt.title("Gradient Alignment vs. Sampler Type")
plt.grid(True, linestyle='--', alpha=0.6)
plt.ylim(-1.1, 1.1)
plt.show()